# Automotive Component Detection — Complete Experiment Pipeline
**Title:** Improving Detection of Visually Challenging Automotive Components:  
**Subtitle:** An Empirical Evaluation of YOLO Training Strategies  
**Author:** Manan Chauhan | GISMA University of Applied Sciences  
**Repository:** https://github.com/manan36chauhan/automotive-component-detection

**All 6 experiments in one notebook — optimised for Kaggle/Colab GPU**

| Exp | What | Variable | Est. Time | Status |
|-----|------|----------|-----------|--------|
| 1 | Baseline YOLOv8s | Default settings | ~45 min | Run this |
| 2 | Architecture: YOLOv8n/s/m | Model capacity | ~2 hrs | Run this |
| 3 | Resolution: 320/640/800 | Input size | ~2 hrs | Run this |
| 4 | Augmentation: none/standard/advanced | Aug. policy | ~2 hrs | Run this |
| 5 | Transfer learning: pretrained vs scratch | Initialisation | ~1.5 hrs | Run this |
| 6 | YOLO26s vs YOLOv8s | Cross-generation | ~1 hr | Run this |

---
## Setup — Run this every session

In [ ]:
!pip install ultralytics roboflow -q

import torch, os, json, time, glob, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import cv2
from collections import Counter
from ultralytics import YOLO
from pathlib import Path
import seaborn as sns
from scipy import stats

# GPU check
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("NO GPU — enable it in Runtime > Change runtime type!")

# Detect platform
if os.path.exists('/kaggle'):
    PLATFORM = 'kaggle'
    BASE_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    PLATFORM = 'colab'
    BASE_DIR = '/content'
else:
    PLATFORM = 'local'
    BASE_DIR = '.'

DATASET_DIR = f'{BASE_DIR}/dataset'
OUTPUT_DIR = f'{BASE_DIR}/results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Platform: {PLATFORM}")
print(f"Dataset: {DATASET_DIR}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# Download dataset
# IMPORTANT: Use environment variable for API key — do NOT hardcode
# Set your key: In Colab, use Secrets; locally, export ROBOFLOW_API_KEY=yourkey

if os.path.exists(f'{DATASET_DIR}/data.yaml'):
    print("Dataset already exists.")
else:
    from roboflow import Roboflow
    api_key = os.environ.get('ROBOFLOW_API_KEY')
    if not api_key:
        # Fallback: prompt user
        api_key = input("Enter your Roboflow API key: ")
    rf = Roboflow(api_key=api_key)
    project = rf.workspace("team-data").project("car-parts-ybiev")
    version = project.version(1)
    dataset = version.download("yolov8", location=DATASET_DIR)

# Find data.yaml (Roboflow sometimes nests it)
DATA_YAML = None
for root, dirs, files in os.walk(DATASET_DIR):
    if 'data.yaml' in files:
        DATA_YAML = os.path.join(root, 'data.yaml')
        break

if DATA_YAML:
    print(f"data.yaml found: {DATA_YAML}")
    with open(DATA_YAML, 'r') as f:
        config = yaml.safe_load(f)
    CLASS_NAMES = config['names']
    NUM_CLASSES = len(CLASS_NAMES)
    print(f"Classes: {NUM_CLASSES}")
    print(f"Class names: {CLASS_NAMES}")
else:
    print("ERROR: data.yaml not found!")

---
## Dataset Exploration (Chapter 4 support)
Analyse class distribution, image statistics, and sample visualisation before training.

In [ ]:
# Count instances per class across train/val/test splits
def count_class_instances(split_dir):
    """Count instances per class from YOLO-format label files."""
    counts = Counter()
    label_dir = os.path.join(split_dir, 'labels')
    if not os.path.exists(label_dir):
        # Try nested path
        for root, dirs, files in os.walk(DATASET_DIR):
            if root.endswith(f'{os.path.basename(split_dir)}/labels'):
                label_dir = root
                break
    
    if os.path.exists(label_dir):
        for f in glob.glob(os.path.join(label_dir, '*.txt')):
            with open(f, 'r') as fh:
                for line in fh:
                    parts = line.strip().split()
                    if parts:
                        counts[int(parts[0])] += 1
    return counts

# Count images per split
split_info = {}
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(DATASET_DIR, split, 'images')
    if not os.path.exists(img_dir):
        # Try nested path
        for root, dirs, files in os.walk(DATASET_DIR):
            if root.endswith(f'{split}/images'):
                img_dir = root
                break
    n_images = len(glob.glob(os.path.join(img_dir, '*'))) if os.path.exists(img_dir) else 0
    split_info[split] = n_images
    print(f"{split}: {n_images} images")

print(f"\nTotal images: {sum(split_info.values())}")

# Count train class instances
train_counts = count_class_instances(os.path.join(DATASET_DIR, 'train'))
class_sample_counts = {}
for idx, name in enumerate(CLASS_NAMES):
    class_sample_counts[name] = train_counts.get(idx, 0)

print(f"\nTotal training annotations: {sum(train_counts.values())}")
print(f"\nClass instance counts (training set):")
for name, count in sorted(class_sample_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:<25} {count:>5}")

In [ ]:
# Class distribution histogram (Figure for Chapter 4)
sorted_classes = sorted(class_sample_counts.items(), key=lambda x: x[1], reverse=True)
names_sorted = [c[0] for c in sorted_classes]
counts_sorted = [c[1] for c in sorted_classes]

fig, ax = plt.subplots(figsize=(10, 14))
colors = ['#E24B4A' if n in ['IGNITION COIL', 'GAS CAP', 'DISTRIBUTOR', 'OVERFLOW TANK', 'OIL PRESSURE SENSOR'] 
          else '#4A90D9' for n in names_sorted]
ax.barh(range(len(names_sorted)), counts_sorted, color=colors)
ax.set_yticks(range(len(names_sorted)))
ax.set_yticklabels(names_sorted, fontsize=8)
ax.set_xlabel('Number of Training Instances')
ax.invert_yaxis()
plt.title('Class Distribution in Training Set\n(red = hardest classes from baseline)', fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR}/class_distribution.png")

In [ ]:
# Image resolution statistics
train_img_dir = os.path.join(DATASET_DIR, 'train', 'images')
if not os.path.exists(train_img_dir):
    for root, dirs, files in os.walk(DATASET_DIR):
        if root.endswith('train/images'):
            train_img_dir = root
            break

widths, heights = [], []
sample_imgs = glob.glob(os.path.join(train_img_dir, '*'))[:500]  # Sample for speed
for img_path in sample_imgs:
    try:
        img = cv2.imread(img_path)
        if img is not None:
            h, w = img.shape[:2]
            widths.append(w)
            heights.append(h)
    except:
        pass

print(f"Image resolution statistics (sampled {len(widths)} images):")
print(f"  Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}, median: {np.median(widths):.0f}")
print(f"  Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}, median: {np.median(heights):.0f}")

In [ ]:
# Sample images per class (visual reference for Chapter 4)
# Show 1 sample image for each of the 5 hard classes
HARD_CLASSES = ['IGNITION COIL', 'GAS CAP', 'DISTRIBUTOR', 'OVERFLOW TANK', 'OIL PRESSURE SENSOR']
hard_class_indices = [CLASS_NAMES.index(c) for c in HARD_CLASSES if c in CLASS_NAMES]

print(f"Hard class indices: {dict(zip(HARD_CLASSES, hard_class_indices))}")
print(f"Hard class training samples: {[(c, class_sample_counts.get(c, 0)) for c in HARD_CLASSES]}")

---
## Training Helper Function
Extracts **full per-class metrics** (mAP50, mAP50-95, precision, recall) for all 50 classes.

In [ ]:
# Shared training settings
EPOCHS = 50
PATIENCE = 10
SEED = 42  # Fixed seed for reproducibility

def train_and_evaluate(model_name, experiment_name, extra_args={}):
    """Train a model and return full per-class results.
    
    Returns dict with:
      - overall: {mAP50, mAP50_95, precision, recall}
      - per_class: {class_name: {mAP50, mAP50_95, precision, recall}} for all 50 classes
      - training_time_min: float
    """
    model = YOLO(model_name)
    start = time.time()
    
    train_args = {
        'data': DATA_YAML,
        'epochs': EPOCHS,
        'patience': PATIENCE,
        'batch': 16,
        'imgsz': 640,
        'device': 0,
        'save': True,
        'plots': True,  # Generates confusion matrices automatically
        'project': f'{OUTPUT_DIR}/runs',
        'name': experiment_name,
        'exist_ok': True,
        'seed': SEED,
    }
    train_args.update(extra_args)
    
    model.train(**train_args)
    train_time = time.time() - start
    
    # Evaluate on test set
    best_path = f'{OUTPUT_DIR}/runs/{experiment_name}/weights/best.pt'
    best = YOLO(best_path)
    imgsz = extra_args.get('imgsz', 640)
    val = best.val(data=DATA_YAML, split='test', imgsz=imgsz, plots=True)
    
    # Extract FULL per-class results
    results = {
        'overall': {
            'mAP50': round(float(val.box.map50), 4),
            'mAP50_95': round(float(val.box.map), 4),
            'precision': round(float(val.box.mp), 4),
            'recall': round(float(val.box.mr), 4),
        },
        'per_class': {},
        'training_time_min': round(train_time / 60, 1),
    }
    
    for i, name in enumerate(CLASS_NAMES):
        if i < len(val.box.ap50):
            results['per_class'][name] = {
                'mAP50': round(float(val.box.ap50[i]), 4),
                'mAP50_95': round(float(val.box.ap[i]), 4) if i < len(val.box.ap) else None,
                'precision': round(float(val.box.p[i]), 4) if i < len(val.box.p) else None,
                'recall': round(float(val.box.r[i]), 4) if i < len(val.box.r) else None,
            }
    
    # Print summary with hard classes
    print(f"\n{'='*60}")
    print(f"{experiment_name} | Overall mAP@0.5: {results['overall']['mAP50']} | "
          f"mAP@0.5:0.95: {results['overall']['mAP50_95']} | Time: {results['training_time_min']}min")
    print(f"{'='*60}")
    print(f"{'Hard Class':<25} {'mAP50':>8} {'mAP50-95':>10} {'Prec':>8} {'Recall':>8}")
    print('-'*62)
    for cls in HARD_CLASSES:
        if cls in results['per_class']:
            d = results['per_class'][cls]
            print(f"{cls:<25} {d['mAP50']:>8} {d.get('mAP50_95', 'N/A'):>10} "
                  f"{d.get('precision', 'N/A'):>8} {d.get('recall', 'N/A'):>8}")
    
    return results

print("Helper function ready.")
print(f"Fixed seed: {SEED} (for reproducibility across all experiments)")
print(f"Note: For full statistical rigour, each experiment should ideally be")
print(f"      run 3x with different seeds. This uses a fixed seed due to")
print(f"      computational constraints, which is noted as a limitation.")

---
## Experiment 1: Baseline (YOLOv8s)
**Establishes reference performance and identifies the 5 hardest classes.**  
Model: YOLOv8s | Resolution: 640px | Default settings | Est: ~45 min

In [ ]:
exp1_results = train_and_evaluate(
    model_name='yolov8s.pt',
    experiment_name='exp1_yolov8s_baseline'
)

with open(f'{OUTPUT_DIR}/exp1_baseline.json', 'w') as f:
    json.dump(exp1_results, f, indent=2)

print(f"\nBaseline mAP@0.5: {exp1_results['overall']['mAP50']}")
print(f"Baseline mAP@0.5:0.95: {exp1_results['overall']['mAP50_95']}")

---
## Experiment 2: Architecture Comparison (RQ2a)
**Does increased model capacity improve detection of hard classes?**

Testing: YOLOv8n (3.2M params) vs YOLOv8s (11.2M) vs YOLOv8m (25.9M) | Est: ~2 hours

In [ ]:
exp2_results = {}

for model_size in ['yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt']:
    size_label = model_size.replace('yolov8', '').replace('.pt', '')
    print(f"\n>>> Architecture: YOLOv8{size_label}")
    
    exp2_results[f'yolov8{size_label}'] = train_and_evaluate(
        model_name=model_size,
        experiment_name=f'exp2_yolov8{size_label}'
    )

with open(f'{OUTPUT_DIR}/exp2_architecture.json', 'w') as f:
    json.dump(exp2_results, f, indent=2)

print("\n\nARCHITECTURE COMPARISON:")
for name, data in exp2_results.items():
    print(f"  {name}: mAP@0.5 = {data['overall']['mAP50']}, "
          f"mAP@0.5:0.95 = {data['overall']['mAP50_95']}, "
          f"Time = {data['training_time_min']}min")

---
## Experiment 3: Resolution Impact (RQ2a)
**Does higher input resolution improve detection of visually challenging components?**

Testing: 320px vs 640px vs 800px | Model: YOLOv8s | Est: ~2 hours

In [ ]:
exp3_results = {}

for imgsz in [320, 640, 800]:
    print(f"\n>>> Resolution: {imgsz}px")
    batch = 16 if imgsz <= 640 else 8  # Reduce batch for higher res
    
    exp3_results[str(imgsz)] = train_and_evaluate(
        model_name='yolov8s.pt',
        experiment_name=f'exp3_res{imgsz}',
        extra_args={'imgsz': imgsz, 'batch': batch}
    )

with open(f'{OUTPUT_DIR}/exp3_resolution.json', 'w') as f:
    json.dump(exp3_results, f, indent=2)

print("\n\nRESOLUTION COMPARISON:")
for res, data in exp3_results.items():
    print(f"  {res}px: mAP@0.5 = {data['overall']['mAP50']}, "
          f"mAP@0.5:0.95 = {data['overall']['mAP50_95']}, "
          f"Time = {data['training_time_min']}min")

---
## Experiment 4: Augmentation Strategies (RQ3)
**Which augmentations help the most for hard-to-detect parts?**

Testing: No augmentation vs Standard vs Advanced (mosaic+mixup) | Est: ~2 hours

In [ ]:
aug_configs = {
    'no_aug': {
        'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0,
        'degrees': 0.0, 'translate': 0.0, 'scale': 0.0,
        'fliplr': 0.0, 'mosaic': 0.0, 'mixup': 0.0,
    },
    'standard_aug': {
        'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4,
        'degrees': 10.0, 'translate': 0.1, 'scale': 0.5,
        'fliplr': 0.5, 'mosaic': 0.0, 'mixup': 0.0,
    },
    'advanced_aug': {
        'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4,
        'degrees': 15.0, 'translate': 0.1, 'scale': 0.5,
        'fliplr': 0.5, 'mosaic': 1.0, 'mixup': 0.15,
    },
}

exp4_results = {}

for name, params in aug_configs.items():
    print(f"\n>>> Augmentation: {name}")
    exp4_results[name] = train_and_evaluate(
        model_name='yolov8s.pt',
        experiment_name=f'exp4_{name}',
        extra_args=params
    )

with open(f'{OUTPUT_DIR}/exp4_augmentation.json', 'w') as f:
    json.dump(exp4_results, f, indent=2)

print("\n\nAUGMENTATION COMPARISON:")
for name, data in exp4_results.items():
    print(f"  {name}: mAP@0.5 = {data['overall']['mAP50']}, "
          f"mAP@0.5:0.95 = {data['overall']['mAP50_95']}")

---
## Experiment 5: Transfer Learning vs From Scratch (RQ2b)
**Does COCO pretraining help or hurt hard classes?**

Testing: Pretrained weights vs Random initialisation | Est: ~1.5 hours

In [ ]:
exp5_results = {}

# Pretrained (COCO weights)
print(">>> Transfer learning: pretrained (COCO)")
exp5_results['pretrained'] = train_and_evaluate(
    model_name='yolov8s.pt',
    experiment_name='exp5_pretrained'
)

# From scratch (random weights)
print("\n>>> Transfer learning: from scratch")
exp5_results['from_scratch'] = train_and_evaluate(
    model_name='yolov8s.yaml',  # .yaml = architecture only, no weights
    experiment_name='exp5_from_scratch',
    extra_args={'epochs': 80, 'patience': 15}  # More epochs needed to converge
)

with open(f'{OUTPUT_DIR}/exp5_transfer_learning.json', 'w') as f:
    json.dump(exp5_results, f, indent=2)

print("\n\nTRANSFER LEARNING COMPARISON:")
for name, data in exp5_results.items():
    print(f"  {name}: mAP@0.5 = {data['overall']['mAP50']}, "
          f"mAP@0.5:0.95 = {data['overall']['mAP50_95']}, "
          f"Time = {data['training_time_min']}min")

---
## Experiment 6: YOLO26 vs YOLOv8 (Cross-Generational Comparison)
**Does the latest YOLO with Small-Target-Aware training improve hard classes?**

YOLO26 released Sep 2025 — very few studies have tested it on domain-specific data.  
This is one of the first empirical comparisons on an automotive component dataset. | Est: ~1 hour

In [ ]:
exp6_results = {}

print(">>> YOLO26s training")
exp6_results['yolo26s'] = train_and_evaluate(
    model_name='yolo26s.pt',
    experiment_name='exp6_yolo26s'
)

with open(f'{OUTPUT_DIR}/exp6_yolo26.json', 'w') as f:
    json.dump(exp6_results, f, indent=2)

print(f"\nYOLO26s mAP@0.5: {exp6_results['yolo26s']['overall']['mAP50']}")
print(f"YOLO26s mAP@0.5:0.95: {exp6_results['yolo26s']['overall']['mAP50_95']}")

---
## Results Analysis — Run after ALL experiments complete
### Overall Experiment Comparison

In [ ]:
# Load all results
all_results = {}
for f_path in sorted(glob.glob(f'{OUTPUT_DIR}/exp*.json')):
    exp_name = os.path.basename(f_path).replace('.json', '')
    with open(f_path, 'r') as f:
        all_results[exp_name] = json.load(f)
    print(f"Loaded: {exp_name}")

# Build comparison table
rows = []
for exp_name, exp_data in all_results.items():
    if 'overall' in exp_data:
        rows.append({'Experiment': exp_name, **exp_data['overall']})
    else:
        for variant, vdata in exp_data.items():
            if isinstance(vdata, dict) and 'overall' in vdata:
                rows.append({'Experiment': f"{exp_name}/{variant}", **vdata['overall']})

df = pd.DataFrame(rows)
print("\n" + "="*75)
print("COMPLETE EXPERIMENT COMPARISON")
print("="*75)
print(df.to_string(index=False))
df.to_csv(f'{OUTPUT_DIR}/all_experiments_comparison.csv', index=False)

### Hard Classes Comparison Across All Experiments

In [ ]:
# Hard classes comparison across ALL experiments (mAP50, precision, recall)
hard_data_map50 = {}
hard_data_prec = {}
hard_data_rec = {}

def extract_hard_class_data(exp_data, label):
    if 'per_class' in exp_data:
        for cls in HARD_CLASSES:
            if cls in exp_data['per_class']:
                d = exp_data['per_class'][cls]
                hard_data_map50.setdefault(cls, {})[label] = d['mAP50']
                if d.get('precision') is not None:
                    hard_data_prec.setdefault(cls, {})[label] = d['precision']
                if d.get('recall') is not None:
                    hard_data_rec.setdefault(cls, {})[label] = d['recall']

for exp_name, exp_data in all_results.items():
    if 'per_class' in exp_data:
        extract_hard_class_data(exp_data, exp_name)
    else:
        for variant, vdata in exp_data.items():
            if isinstance(vdata, dict) and 'per_class' in vdata:
                extract_hard_class_data(vdata, f"{exp_name}/{variant}")

if hard_data_map50:
    df_hard = pd.DataFrame(hard_data_map50).T
    print("\nHARD CLASSES — mAP@0.5 ACROSS ALL EXPERIMENTS:")
    print(df_hard.to_string())
    df_hard.to_csv(f'{OUTPUT_DIR}/hard_classes_map50.csv')
    
    if hard_data_prec:
        df_prec = pd.DataFrame(hard_data_prec).T
        print("\nHARD CLASSES — PRECISION ACROSS ALL EXPERIMENTS:")
        print(df_prec.to_string())
        df_prec.to_csv(f'{OUTPUT_DIR}/hard_classes_precision.csv')
    
    if hard_data_rec:
        df_rec = pd.DataFrame(hard_data_rec).T
        print("\nHARD CLASSES — RECALL ACROSS ALL EXPERIMENTS:")
        print(df_rec.to_string())
        df_rec.to_csv(f'{OUTPUT_DIR}/hard_classes_recall.csv')

In [ ]:
# Hard classes chart (mAP@0.5)
if hard_data_map50:
    fig, axes = plt.subplots(1, len(HARD_CLASSES), figsize=(22, 6))
    for i, cls in enumerate(HARD_CLASSES):
        if cls in hard_data_map50:
            exps = list(hard_data_map50[cls].keys())
            vals = list(hard_data_map50[cls].values())
            colors = ['#E24B4A' if v < 0.85 else '#EF9F27' if v < 0.92 else '#97C459' for v in vals]
            axes[i].barh(range(len(exps)), vals, color=colors)
            axes[i].set_yticks(range(len(exps)))
            axes[i].set_yticklabels([e.split('/')[-1] for e in exps], fontsize=7)
            axes[i].set_xlim(0, 1)
            axes[i].set_title(cls, fontsize=9, fontweight='bold')
            axes[i].axvline(x=0.85, color='red', linestyle='--', linewidth=0.8, alpha=0.7)
            # Add value labels
            for j, v in enumerate(vals):
                axes[i].text(v + 0.01, j, f'{v:.3f}', va='center', fontsize=6)
    plt.suptitle('Hard Classes: mAP@0.5 Across All Experiments\n(red = <0.85 | orange = 0.85-0.92 | green = >0.92)',
                 fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/hard_classes_chart.png', dpi=150, bbox_inches='tight')
    plt.show()

### Per-Class Baseline Chart (Motivational Figure for Chapter 1)

In [ ]:
# Generate motivational figure from actual baseline results
# If exp1 results are loaded, use them; otherwise use hardcoded baseline values
if 'exp1_baseline' in all_results and 'per_class' in all_results['exp1_baseline']:
    baseline_per_class = {k: v['mAP50'] for k, v in all_results['exp1_baseline']['per_class'].items()}
else:
    # Fallback: hardcoded values from previous runs
    baseline_per_class = {
        'CLUTCH PLATE': 0.994, 'BATTERY': 0.993, 'TORQUE CONVERTER': 0.993,
        'PRESSURE PLATE': 0.992, 'AIR COMPRESSOR': 0.991,
        'INSTRUMENT CLUSTER': 0.991, 'COIL SPRING': 0.986, 'BRAKE ROTOR': 0.986,
        'RADIO': 0.985, 'RADIATOR FAN': 0.984,
        'ENGINE VALVE': 0.981, 'RIM': 0.981, 'CRANKSHAFT': 0.978,
        'WINDOW REGULATOR': 0.978, 'TAILLIGHTS': 0.977,
        'TRANSMISSION': 0.979, 'OXYGEN SENSOR': 0.972, 'THERMOSTAT': 0.966,
        'SPARK PLUG': 0.965, 'VALVE LIFTER': 0.964,
        'BRAKE PAD': 0.963, 'CAMSHAFT': 0.958, 'RADIATOR': 0.958,
        'CARBERATOR': 0.957, 'PISTON': 0.957,
        'FUSE BOX': 0.954, 'ALTERNATOR': 0.954, 'STARTER': 0.950,
        'LEAF SPRING': 0.949, 'WATER PUMP': 0.941,
        'LOWER CONTROL ARM': 0.939, 'IDLER ARM': 0.934, 'HEADLIGHTS': 0.933,
        'OIL FILTER': 0.932, 'FUEL INJECTOR': 0.928,
        'VACUUM BRAKE BOOSTER': 0.926, 'SPOILER': 0.926, 'CYLINDER HEAD': 0.918,
        'OIL PAN': 0.919, 'SIDE MIRROR': 0.906,
        'ENGINE BLOCK': 0.905, 'RADIATOR HOSE': 0.904, 'SHIFT KNOB': 0.904,
        'MUFFLER': 0.903, 'BRAKE CALIPER': 0.887,
        'OIL PRESSURE SENSOR': 0.848, 'OVERFLOW TANK': 0.810,
        'DISTRIBUTOR': 0.792, 'GAS CAP': 0.717, 'IGNITION COIL': 0.675,
    }

# Sort by performance (descending)
sorted_baseline = sorted(baseline_per_class.items(), key=lambda x: x[1], reverse=True)
names = [c[0] for c in sorted_baseline]
values = [c[1] for c in sorted_baseline]
colors = ['#E24B4A' if v < 0.85 else '#4A90D9' for v in values]

fig, ax = plt.subplots(figsize=(10, 14))
bars = ax.barh(range(len(names)), values, color=colors)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=8)
ax.axvline(x=0.85, color='red', linestyle='--', linewidth=1, label='0.85 threshold')
ax.set_xlabel('mAP@0.5', fontsize=11)
ax.set_xlim(0.6, 1.02)
ax.legend(loc='lower right')
ax.invert_yaxis()

# Add value labels for hard classes
for i, (n, v) in enumerate(sorted_baseline):
    if v < 0.85:
        ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=7, color='#E24B4A', fontweight='bold')

plt.title('Per-Class Detection Performance — Baseline YOLOv8s\n'
          '(red bars = below 0.85 threshold; 32-point gap between best and worst class)',
          fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ch1_motivational_figure.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR}/ch1_motivational_figure.png")

---
### Class Distribution vs Performance Correlation (RQ1 — Key Analysis)
**Does training sample count explain per-class performance, or is visual complexity the primary factor?**

In [ ]:
# Scatter plot: training samples vs mAP@0.5 per class
if baseline_per_class and class_sample_counts:
    common_classes = set(baseline_per_class.keys()) & set(class_sample_counts.keys())
    x_samples = [class_sample_counts[c] for c in common_classes]
    y_map = [baseline_per_class[c] for c in common_classes]
    labels = list(common_classes)
    
    # Compute correlation
    pearson_r, pearson_p = stats.pearsonr(x_samples, y_map)
    spearman_r, spearman_p = stats.spearmanr(x_samples, y_map)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    colors_scatter = ['#E24B4A' if c in HARD_CLASSES else '#4A90D9' for c in common_classes]
    ax.scatter(x_samples, y_map, c=colors_scatter, s=60, alpha=0.8, edgecolors='white', linewidth=0.5)
    
    # Label hard classes
    for c in common_classes:
        if c in HARD_CLASSES:
            ax.annotate(c, (class_sample_counts[c], baseline_per_class[c]),
                       fontsize=7, fontweight='bold', color='#E24B4A',
                       xytext=(5, 5), textcoords='offset points')
    
    ax.axhline(y=0.85, color='red', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('Number of Training Instances', fontsize=11)
    ax.set_ylabel('mAP@0.5 (Baseline YOLOv8s)', fontsize=11)
    ax.set_title(f'Training Sample Count vs Detection Performance\n'
                 f'Pearson r={pearson_r:.3f} (p={pearson_p:.4f}) | '
                 f'Spearman r={spearman_r:.3f} (p={spearman_p:.4f})',
                 fontsize=11)
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/correlation_samples_vs_map.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nCorrelation Analysis:")
    print(f"  Pearson r  = {pearson_r:.4f} (p = {pearson_p:.4f})")
    print(f"  Spearman r = {spearman_r:.4f} (p = {spearman_p:.4f})")
    if abs(pearson_r) < 0.3:
        print(f"  Interpretation: Weak correlation — visual complexity likely matters")
        print(f"  more than sample count for explaining per-class performance.")
    elif abs(pearson_r) < 0.6:
        print(f"  Interpretation: Moderate correlation — both sample count and")
        print(f"  visual complexity contribute to per-class performance variation.")
    else:
        print(f"  Interpretation: Strong correlation — sample count is a major")
        print(f"  factor in per-class performance.")
else:
    print("Run dataset exploration and baseline experiment first.")

### Confusion Matrix Analysis (RQ1)

In [ ]:
# Display confusion matrix from baseline experiment
# Ultralytics saves this automatically during training and validation
confusion_paths = [
    f'{OUTPUT_DIR}/runs/exp1_yolov8s_baseline/confusion_matrix_normalized.png',
    f'{OUTPUT_DIR}/runs/exp1_yolov8s_baseline/confusion_matrix.png',
]

for cm_path in confusion_paths:
    if os.path.exists(cm_path):
        img = cv2.imread(cm_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(figsize=(16, 14))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(os.path.basename(cm_path).replace('.png', '').replace('_', ' ').title(),
                    fontsize=14)
        plt.tight_layout()
        plt.show()
        print(f"\nLoaded: {cm_path}")
    else:
        print(f"Not found: {cm_path}")
        print("Run Experiment 1 first. Ultralytics generates confusion matrices automatically.")

print("\nNote: Examine the confusion matrix to identify which class pairs are")
print("most frequently confused. This directly addresses RQ1 by revealing")
print("inter-class similarity as a difficulty factor.")

### Full Per-Class Results Table (for Chapter 5)

In [ ]:
# Complete per-class table from baseline
if 'exp1_baseline' in all_results and 'per_class' in all_results['exp1_baseline']:
    pc = all_results['exp1_baseline']['per_class']
    pc_rows = []
    for name, metrics in sorted(pc.items(), key=lambda x: x[1]['mAP50']):
        row = {'Class': name}
        row.update(metrics)
        row['train_samples'] = class_sample_counts.get(name, 'N/A')
        pc_rows.append(row)
    
    df_pc = pd.DataFrame(pc_rows)
    print("\nFULL PER-CLASS RESULTS (sorted by mAP@0.5, ascending):")
    print(df_pc.to_string(index=False))
    df_pc.to_csv(f'{OUTPUT_DIR}/full_per_class_baseline.csv', index=False)
    print(f"\nSaved: {OUTPUT_DIR}/full_per_class_baseline.csv")
else:
    print("Run Experiment 1 first.")

---
### Detection Visualisations — Sample Predictions

In [ ]:
# Detection visualisations — sample predictions with bounding boxes
best_model_path = None
for candidate in [
    f'{OUTPUT_DIR}/runs/exp6_yolo26s/weights/best.pt',
    f'{OUTPUT_DIR}/runs/exp1_yolov8s_baseline/weights/best.pt',
]:
    if os.path.exists(candidate):
        best_model_path = candidate
        break

if best_model_path:
    vis_model = YOLO(best_model_path)
    test_images = glob.glob(f'{DATASET_DIR}/test/images/*')[:12]
    
    if not test_images:
        test_images = glob.glob(f'{DATASET_DIR}/*/test/images/*')[:12]
    
    if test_images:
        results = vis_model.predict(source=test_images, conf=0.25, save=True,
            project=f'{OUTPUT_DIR}/visualisations', name='detections', line_width=2)
        
        fig, axes = plt.subplots(3, 4, figsize=(20, 15))
        for ax, r in zip(axes.flat, results):
            img = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.axis('off')
        plt.suptitle('Sample Detection Results', fontsize=14)
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/sample_detections.png', dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print("No test images found.")
else:
    print("No trained model found. Run experiments first.")

### Error Analysis — False Positive/Negative Examples for Hard Classes

In [ ]:
# Error analysis: Show predictions on hard-class test images
# This identifies WHAT the model gets wrong on the hardest classes

if best_model_path:
    vis_model = YOLO(best_model_path)
    hard_indices = [CLASS_NAMES.index(c) for c in HARD_CLASSES if c in CLASS_NAMES]
    
    # Find test images containing hard classes
    test_label_dir = os.path.join(DATASET_DIR, 'test', 'labels')
    if not os.path.exists(test_label_dir):
        for root, dirs, files in os.walk(DATASET_DIR):
            if root.endswith('test/labels'):
                test_label_dir = root
                break
    
    test_img_dir = os.path.join(DATASET_DIR, 'test', 'images')
    if not os.path.exists(test_img_dir):
        for root, dirs, files in os.walk(DATASET_DIR):
            if root.endswith('test/images'):
                test_img_dir = root
                break
    
    hard_images = []
    if os.path.exists(test_label_dir):
        for lbl_file in glob.glob(os.path.join(test_label_dir, '*.txt')):
            with open(lbl_file, 'r') as f:
                classes_in_file = set()
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        classes_in_file.add(int(parts[0]))
                if classes_in_file & set(hard_indices):
                    img_name = os.path.basename(lbl_file).replace('.txt', '.jpg')
                    img_path = os.path.join(test_img_dir, img_name)
                    if not os.path.exists(img_path):
                        img_path = os.path.join(test_img_dir, img_name.replace('.jpg', '.png'))
                    if os.path.exists(img_path):
                        hard_images.append(img_path)
    
    if hard_images:
        sample = hard_images[:8]
        results = vis_model.predict(source=sample, conf=0.25)
        
        cols = min(4, len(sample))
        rows_count = (len(sample) + cols - 1) // cols
        fig, axes = plt.subplots(rows_count, cols, figsize=(5*cols, 5*rows_count))
        if rows_count == 1:
            axes = [axes] if cols == 1 else axes
        axes_flat = np.array(axes).flatten()
        
        for ax, r in zip(axes_flat, results):
            img = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.axis('off')
        for ax in axes_flat[len(results):]:
            ax.axis('off')
        
        plt.suptitle('Predictions on Images Containing Hard Classes\n'
                     '(IGNITION COIL, GAS CAP, DISTRIBUTOR, OVERFLOW TANK, OIL PRESSURE SENSOR)',
                     fontsize=12)
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/hard_class_predictions.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Found {len(hard_images)} test images containing hard classes.")
    else:
        print("No test images with hard classes found.")
else:
    print("No model found. Run experiments first.")

### ONNX Export — Deployment Prototype

In [ ]:
# ONNX Export — prototype demonstration for Chapter 6 (Future Work)
if best_model_path:
    export_model = YOLO(best_model_path)
    onnx_path = export_model.export(format='onnx', imgsz=640)
    print(f"Model exported to ONNX: {onnx_path}")
    print(f"File size: {os.path.getsize(onnx_path) / (1024*1024):.1f} MB")
    print("\nThis demonstrates deployment readiness.")
    print("Mention in Chapter 4 (Implementation) and Chapter 6 (Future Work).")
else:
    print("No model to export. Run experiments first.")

---
### Final Summary

In [ ]:
# Final summary of all generated files
print("="*70)
print("ALL FILES GENERATED")
print("="*70)
for f in sorted(glob.glob(f'{OUTPUT_DIR}/*.*')):
    size = os.path.getsize(f)
    print(f"  {os.path.basename(f)} ({size//1024}KB)")
print(f"\nRun folders:")
for d in sorted(glob.glob(f'{OUTPUT_DIR}/runs/*')):
    if os.path.isdir(d):
        print(f"  {os.path.basename(d)}/")
print(f"\nDownload the entire '{OUTPUT_DIR}' folder to your local machine.")
print("Then push results to GitHub and use the charts in your dissertation.")
print("\nKey outputs for dissertation chapters:")
print("  Chapter 1: ch1_motivational_figure.png")
print("  Chapter 4: class_distribution.png")
print("  Chapter 5: correlation_samples_vs_map.png, hard_classes_chart.png,")
print("             hard_class_predictions.png, full_per_class_baseline.csv,")
print("             all_experiments_comparison.csv, confusion matrices in runs/")
print("  Chapter 6: ONNX export demonstration")